# StatGuard-KR 파이썬 빠른 시작

AI가 데이터 분석으로 쓴 답변 속 숫자를, 같은 CSV로 **LLM 없이** 다시 계산해 검증합니다. 노트북·스크립트에서 함수 하나로 씁니다. 인터넷 연결이나 API 키가 필요 없고, 데이터를 어디에도 저장하지 않습니다.

이 노트북의 출력은 저장소의 엔진으로 실제 실행한 결과입니다.

In [1]:
import sys
sys.path.insert(0, '../src')  # 저장소의 src 폴더를 경로에 추가
from when2warn.statguard.api import verify, check_calculations

## 1. AI 답변 검증

시애틀 2023년 일별 기상(NOAA) CSV와, 실험에서 AI(gpt-6-luna)가 실제로 쓴 답변입니다. `units`에는 숫자 열의 단위를 직접 적습니다(자동 추정하지 않음).

In [2]:
csv = '../experiments/statguard_benchmark_v0.2/datasets/06_시애틀_2023_일별기상.csv'
answer = ('2023년 연간 강수량 합계는 888.4mm입니다. 2023년 12월 강수량 합계는 215.4mm입니다. '
          '따라서 12월 강수량은 연간 합계의 약 24.25%입니다.')
report = verify(csv, answer, units={'강수량': 'mm', '최고기온': '℃', '최저기온': '℃'})
print(report.text())

StatGuard (statguard-rule-extractor-v0.7.1) — 계산 일치 1 · 계산 불일치 2 · 보류 0
[계산 불일치] 2023년 연간 강수량 합계는 888.4mm입니다.
    답변 888.4mm / 재계산 888.9 mm (sum(강수량) = 888.9)
    왜 틀렸을까(확신 중간): 행 하나를 빠뜨림 — 365개 행 중 ‘2023-01-04’ 행(값 0.5)을 빼고 합계를 구하면 888.4mm로 답변의 숫자와 같습니다. 같은 값이 나오는 행이 14개라 어느 행인지는 특정할 수 없습니다.
[계산 일치] 2023년 12월 강수량 합계는 215.4mm입니다.
    답변 215.4mm / 재계산 215.4 mm (sum(강수량) = 215.4)
[계산 불일치] 따라서 12월 강수량은 연간 합계의 약 24.25%입니다.
    답변 24.25% / 재계산 24.23219709753628079649004387445157 % (215.4 / 888.9 * 100)
    왜 틀렸을까(확신 높음): 앞의 틀린 숫자를 그대로 사용 — 답변이 다른 문장에서 틀리게 말한 ‘888.4mm’라는 값을 분모로 쓰면 24.25%로 답변의 숫자와 같습니다. 앞 문장의 오류가 이 계산으로 번졌습니다.


‘왜 틀렸을까’는 오류 역추적입니다. 24.25%는 앞 문장에서 틀리게 말한 888.4mm를 분모로 쓴 값과 정확히 같습니다.

한계: 한 문장에 숫자를 여러 개 쓰면(“12월은 215.4mm이고 연간은 888.4mm”) 범위를 잘못 연결할 수 있어, 숫자마다 문장을 나누는 편이 정확합니다.

## 2. 결과를 코드에서 쓰기

In [3]:
print(report.summary)
print('문제 없음' if report.ok else '확인 필요')
for c in report.claims:
    print(c['status'], c['claimed']['text'] if c.get('claimed') else '', (c.get('computed') or {}).get('value', '')[:10])

{'VERIFIED': 1, 'CONTRADICTED': 2, 'UNCERTAIN': 0, 'INSUFFICIENT_EVIDENCE': 0}
확인 필요
CONTRADICTED 888.4mm 888.9
VERIFIED 215.4mm 215.4
CONTRADICTED 24.25% 24.2321970


## 3. 기간 범위

“하반기(7~12월)”, “상반기”, “1분기”, “2016~2020년” 같은 범위는 해당 행만 골라 계산합니다.

In [4]:
r = verify(csv, '2023년 하반기(7~12월) 강수량 합계는 532.1mm입니다.', units={'강수량': 'mm'})
print(r.text())
print(r.claims[0]['spec']['value_sets'])

StatGuard (statguard-rule-extractor-v0.7.1) — 계산 일치 1 · 계산 불일치 0 · 보류 0
[계산 일치] 2023년 하반기(7~12월) 강수량 합계는 532.1mm입니다.
    답변 532.1mm / 재계산 532.1 mm (sum(강수량) = 532.1)
{'월': ['7', '8', '9', '10', '11', '12']}


## 4. CSV 없이 답변 속 계산만 검산

자료가 없어도 답변이 직접 적은 계산식은 다시 계산할 수 있습니다.

In [5]:
result = check_calculations('월별 강수량을 더하면 78.5 + 60.1 + 65.3 + 98.1 + 23.8 + 31.0 + 2.3 + 6.8 + 87.3 + 73.3 + 147.0 + 215.4 = 789.0 mm입니다.')
for c in result['checks']:
    print(c['status'], c['expression'], '=', c['stated'], '→ 재계산', c['recomputed'])

INCONSISTENT 78.5 + 60.1 + 65.3 + 98.1 + 23.8 + 31.0 + 2.3 + 6.8 + 87.3 + 73.3 + 147.0 + 215.4 = 789.0 → 재계산 888.9


## 5. 명령줄

```powershell
$env:PYTHONPATH = (Resolve-Path ..\src).Path
python -m when2warn.statguard verify --csv 자료.csv --unit 강수량=mm --answer "2023년 연간 강수량은 888.4mm입니다."
python -m when2warn.statguard check --answer "78.5 + 60.1 = 140.7mm"
```

결과가 문제없으면 종료 코드 0, 계산 불일치나 검산 오류가 있으면 1입니다(자동화 파이프라인에서 쓸 수 있음).